# **Obtener el dataset para entrenamiento del modelo de ReID**

En este notebook, se obtienen los datos para el entrenamiento del modelo de reidentificación.

## **Constantes**

In [ ]:
# Paths a los vídeos de entrenamiento y de query/galería
# Cada carpeta tiene una subcarpeta gt con gt.txt (anotaciones de seguimiento) y otra subcarpeta img1 con los frames
PATH_VIDEO_QUERY_GALLERY = "../data/sportsmot/sportsmot_publish/dataset/train/v_2j7kLB-vEEk_c005"
PATH_VIDEO_TRAIN_2 = "../data/sportsmot/sportsmot_publish/dataset/train/v_4LXTUim5anY_c012"
PATH_VIDEO_TRAIN_1 = "../data/sportsmot/sportsmot_publish/dataset/train/v_-6Os86HzwCs_c001"

UMBRAL_MAXIMO_IOU = 0.50 # Máximo 50% de IoU entre dos cajas de jugadores para considerarlas para el dataset de reid

# Mínimo y máximo de imágenes por jugador para el dataset de reid
MINIMO_IMAGENES_POR_JUGADOR_TRAIN = 30
MAXIMO_IMAGENES_POR_JUGADOR_TRAIN = 80

MINIMO_IMAGENES_POR_JUGADOR_QUERY = 1
MAXIMO_IMAGENES_POR_JUGADOR_QUERY = 4

MINIMO_IMAGENES_POR_JUGADOR_GALLERY = 3
MAXIMO_IMAGENES_POR_JUGADOR_GALLERY = 10

MIN_WIDTH_IMG = 24
MIN_HEIGHT_IMG = 48

# Salida imágenes
PATH_OUTPUT_IMAGES_TRAIN = "./data/train"
PATH_OUTPUT_IMAGES_QUERY = "./data/query"
PATH_OUTPUT_IMAGES_GALLERY = "./data/gallery"
# Crea las carpetas si no existen
import os
os.makedirs(PATH_OUTPUT_IMAGES_TRAIN, exist_ok=True)
os.makedirs(PATH_OUTPUT_IMAGES_QUERY, exist_ok=True)
os.makedirs(PATH_OUTPUT_IMAGES_GALLERY, exist_ok=True)

## **Train split**

Librerías

In [ ]:
#@title: Importar librerías
import os
import csv
from collections import defaultdict
from pathlib import Path

from PIL import Image

from torchvision.ops import box_iou
import torch

Funciones auxiliares

In [ ]:
def parse_gt(gt_file):
    """Obtiene las anotaciones de seguimiento de un archivo y devuelve una lista de (frame, track_id, x, y, w, h)"""
    rows = []
    with open(gt_file, "r") as f:
        reader = csv.reader(f)
        for r in reader:
            if len(r) < 6:
                continue
            frame = int(float(r[0]))
            tid = int(float(r[1])) # Track ID
            x = float(r[2])
            y = float(r[3])
            w = float(r[4])
            h = float(r[5])
            if w <= 0 or h <= 0:
                continue
            if w < MIN_WIDTH_IMG or h < MIN_HEIGHT_IMG:
                print("DESCARTO IMAGEN POR SER MUY PEQUEÑA: w={}, h={}".format(w, h))
                continue
            conf = 1.0
            if len(r) > 6:
                conf = float(r[6])
            if conf <= 0:
                continue
            rows.append((frame, tid, x, y, w, h))
    return rows

def filtrar_por_iou(rows, umbral_iou=UMBRAL_MAXIMO_IOU):
    """Filtra la lista rows para eliminar las imágenes de jugadores con IoU mayor que el umbral"""

    # Agrupa por frame
    frames = defaultdict(list)
    for row in rows:
        frame, tid, x, y, w, h = row
        frames[frame].append((tid, x, y, w, h))
    
    # Filtra por IoU
    filtered_rows = []
    for frame, players in frames.items():
        boxes = torch.tensor([[x, y, x+w, y+h] for _, x, y, w, h in players])
        iou_matrix = box_iou(boxes, boxes)
        keep_indices = set(range(len(players)))
        for i in range(len(players)):
            for j in range(i+1, len(players)):
                if iou_matrix[i,j] > umbral_iou:
                    # Si hay solapamiento, se descartan las dos cajas
                    keep_indices.discard(i)
                    keep_indices.discard(j)
        for idx in keep_indices:
            filtered_rows.append((frame,) + players[idx])
    
    return filtered_rows

def crop_image(image_path, bbox):
    """Recorta la imagen en image_path según el bbox (x, y, w, h) y devuelve la imagen recortada"""
    x, y, w, h = bbox
    with Image.open(image_path) as img:
        cropped_img = img.crop((x, y, x+w, y+h))
    return cropped_img

In [ ]:
def save_cropped_images(rows, video_path, output_dir, min_images_per_player, max_images_per_player, camids):
    """Guarda las imágenes recortadas de los jugadores según las anotaciones en rows"""
    player_images = defaultdict(list)
    camid_A = camids[0]
    camid_B = camids[1]

    max_frame = max(frame for frame, _, _, _, _, _ in rows)
    
    # Agrupa las imágenes por jugador
    for frame, tid, x, y, w, h in rows:
        image_path = os.path.join(video_path, "img1", f"{frame:06d}.jpg")
        player_images[tid].append((image_path, (x, y, w, h), frame))
    
    # Guarda las imágenes recortadas
    for tid, images in player_images.items():
        if len(images) < min_images_per_player:
            continue
        # Si hay más imágenes de las permitidas, se seleccionan de forma uniforme
        images = sorted(images, key=lambda z: z[2])
        if len(images) > max_images_per_player:
            step = len(images) / max_images_per_player
            selected_indices = [int(i * step) for i in range(max_images_per_player)]
            images = [images[i] for i in selected_indices]
        # Se guardan las imágenes recortadas
        for idx, (image_path, bbox, frame) in enumerate(images):
            # La primera mitad de los frames se asigna a camid_A y la segunda mitad a camid_B
            camid = camid_A if frame <= max_frame // 2 else camid_B
            cropped_img = crop_image(image_path, bbox)
            output_path = os.path.join(output_dir, f"c{camid}_{tid}_{idx:02d}.jpg")
            cropped_img.save(output_path)

Obtener train

In [ ]:
train_rows_1 = parse_gt(os.path.join(PATH_VIDEO_TRAIN_1, "gt", "gt.txt"))
train_rows_1_filtered = filtrar_por_iou(train_rows_1)

train_rows_2 = parse_gt(os.path.join(PATH_VIDEO_TRAIN_2, "gt", "gt.txt"))
train_rows_2_filtered = filtrar_por_iou(train_rows_2)

# Para train_rows_2, ponerles un offset a los track_id para que no se solapen con los de train_rows_1
offset_tid = max(tid for _, tid, _, _, _, _ in train_rows_1_filtered) + 1
train_rows_2_filtered_offset = [(frame, tid + offset_tid, x, y, w, h) for frame, tid, x, y, w, h in train_rows_2_filtered]

# Guarda las imágenes recortadas de los vídeos de entrenamiento
# Se usan cámaras distintas para primera y segunda mitad de los frames. Sin embargo, como en entrenamiento al final se usa RandomIdentitySampler, la cámara etiquetada no se usa para nada. Si se usara por ejemplo RandomDomainSampler, sí que aportarían las dos cámaras
save_cropped_images(train_rows_1_filtered, PATH_VIDEO_TRAIN_1, PATH_OUTPUT_IMAGES_TRAIN, MINIMO_IMAGENES_POR_JUGADOR_TRAIN, MAXIMO_IMAGENES_POR_JUGADOR_TRAIN, camids=[0,1])
save_cropped_images(train_rows_2_filtered_offset, PATH_VIDEO_TRAIN_2, PATH_OUTPUT_IMAGES_TRAIN, MINIMO_IMAGENES_POR_JUGADOR_TRAIN, MAXIMO_IMAGENES_POR_JUGADOR_TRAIN, camids=[2,3])

# Crea el txt de train.txt con las rutas de las imágenes y sus track_id
with open(os.path.join(PATH_OUTPUT_IMAGES_TRAIN, "train.txt"), "w") as f:
    image_files = sorted(
        f for f in os.listdir(PATH_OUTPUT_IMAGES_TRAIN)
        if f.endswith(".jpg")
    )
    for filename in image_files:
        parts = filename.replace(".jpg", "").split("_")
        camid = int(parts[0][1:])
        tid = int(parts[1])
        ruta_img = os.path.join(r"..\..\reid\data\train", filename)
        f.write(f"{ruta_img},{tid},{camid}\n")

## **Query/gallery**

Función auxiliar

In [ ]:
def save_cropped_images_query_gallery(rows, video_path, output_dir_query, output_dir_gallery, min_images_per_player_query, max_images_per_player_query, min_images_per_player_gallery, max_images_per_player_gallery):
    """Guarda las imágenes recortadas de los jugadores según las anotaciones en rows para query y gallery"""
    player_images = defaultdict(list)
    
    # Agrupa las imágenes por jugador
    for frame, tid, x, y, w, h in rows:
        image_path = os.path.join(video_path, "img1", f"{frame:06d}.jpg")
        player_images[tid].append((image_path, (x, y, w, h)))
    
    # Guarda las imágenes recortadas
    for tid, images in player_images.items():
        if len(images) < min_images_per_player_query:
            continue

        # Selecciono las imágenes de query y gallery para este jugador. Selecciono primero las de query y luego el resto para gallery, pero selecciono la primera mitad de las imágenes para query y la segunda mitad para gallery, para asegurar que no haya solapamiento entre ambos conjuntos.
        if len(images) < min_images_per_player_query + min_images_per_player_gallery:
            continue

        images_query = images[:len(images)//2]
        images_gallery = images[len(images)//2:]

        # Si hay más imágenes de las permitidas para query, se seleccionan de forma uniforme
        if len(images_query) > max_images_per_player_query:
            step = len(images_query) / max_images_per_player_query
            selected_indices = [int(i * step) for i in range(max_images_per_player_query)]
            images_query = [images_query[i] for i in selected_indices]
        else:
            images_query = images_query
        
        # Guarda las imágenes de query
        for idx, (image_path, bbox) in enumerate(images_query):
            cropped_img = crop_image(image_path, bbox)
            output_path = os.path.join(output_dir_query, f"{tid}_{idx:02d}.jpg")
            cropped_img.save(output_path)

        # Si hay más imágenes de las permitidas para gallery, se seleccionan de forma uniforme
        if len(images_gallery) > max_images_per_player_gallery:
            step = len(images_gallery) / max_images_per_player_gallery
            selected_indices = [int(i * step) for i in range(max_images_per_player_gallery)]
            images_gallery = [images_gallery[i] for i in selected_indices]
        
        # Guarda las imágenes de gallery
        for idx, (image_path, bbox) in enumerate(images_gallery):
            cropped_img = crop_image(image_path, bbox)
            output_path = os.path.join(output_dir_gallery, f"{tid}_{idx:02d}.jpg")
            cropped_img.save(output_path)

Obtener los conjuntos de query/galería

In [ ]:
# Leer gt.txt del vídeo de query/galería
query_gallery_rows = parse_gt(os.path.join(PATH_VIDEO_QUERY_GALLERY, "gt", "gt.txt"))

# Filtra por IoU
query_gallery_rows_filtered = filtrar_por_iou(query_gallery_rows)

# Guardar las imágenes recortadas de los jugadores del vídeo de query/galería
save_cropped_images_query_gallery(query_gallery_rows_filtered, PATH_VIDEO_QUERY_GALLERY, PATH_OUTPUT_IMAGES_QUERY, PATH_OUTPUT_IMAGES_GALLERY, MINIMO_IMAGENES_POR_JUGADOR_QUERY, MAXIMO_IMAGENES_POR_JUGADOR_QUERY, MINIMO_IMAGENES_POR_JUGADOR_GALLERY, MAXIMO_IMAGENES_POR_JUGADOR_GALLERY)

# Crea el txt de query.txt con las rutas de las imágenes y sus track_id
with open(os.path.join(PATH_OUTPUT_IMAGES_QUERY, "query.txt"), "w") as f:
    for filename in os.listdir(PATH_OUTPUT_IMAGES_QUERY):
        if filename.endswith(".jpg"):
            tid = int(filename.split("_")[0])
            camid = 0
            ruta_img = os.path.join(r"..\..\reid\data\query", filename)
            f.write(f"{ruta_img},{tid},{camid}\n")
 
# Crea el txt de gallery.txt con las rutas de las imágenes y sus track_id
with open(os.path.join(PATH_OUTPUT_IMAGES_GALLERY, "gallery.txt"), "w") as f:
    for filename in os.listdir(PATH_OUTPUT_IMAGES_GALLERY):
        if filename.endswith(".jpg"):
            tid = int(filename.split("_")[0])
            camid = 1
            ruta_img = os.path.join(r"..\..\reid\data\gallery", filename)
            f.write(f"{ruta_img},{tid},{camid}\n")